# Hockey-Reference Scrape, Cleaning, and Final Dataset

Repo: `ewnike/NHL-Beyond-27` — *MADS Milestone I Project*  
Python: 3.13.7 (pyenv env: `nhl_beyond27-3.13.7`)  
Editors/Tools: VSCode, Git/GitHub, Postgres + pgAdmin  
Logs: written via `log_utils.py` (default `logs/`)

**This notebook covers:**
1) What was scraped from Hockey-Reference (Standard + EV) and why
2) Normalization and TOI parsing rules (avg vs. total)
3) (Optional) Running the scrapers to reproduce raw CSVs
4) Building normalized outputs and the merged final table
5) Curating the final (drop goalies, compute EV minutes, trim columns)
6) (Optional) Cap-hit merge
7) (Optional) Load curated data into Postgres
8) Validation/diffs and lessons learned

## Scrape scope

- Seasons: 2013–14 → 2024–25  
- Strength: EV/5v5 focus (plus “Standard” season pages)  
- Sources (per season):
  - Standard skater pages (all situations)
  - Even-strength time-on-ice pages

## Normalization rules

- Column headers → snake_case (e.g., `CF% Rel` → `cf_rel`)  
- `player`: lowercase, trim, collapse whitespace  
- `tm`: uppercase; recognize `TOT`, `2TM`, `3TM`, etc.  
- TOI parsing understands:
  - Per-game `mm:ss` (multiply by GP)
  - Season total `mmmm:ss`
  - Rare `HH:MM:SS`  
- Use `time_utils.compute_total_toi` and `time_utils.seconds_to_hms`.

## Multi-team seasons & merge policy

- Standard:
  - If an `nTM` row exists (e.g., `2TM`), keep that as `tm='TOT'` and drop per-team rows.
  - If no `nTM`, synthesize a `TOT` row by summing numerics (mode position for `pos`).
- Even strength:
  - Do **not** collapse by team in files; compute per-game → season totals.
  - For the merge only, aggregate EV by `(player, season)` to avoid row multiplication (sum TOI seconds; TOI-weighted CF% if available).

## Order of operations

- `scrap_hockey_ref_player.py` — scrape Standard pages per season → `data/seasons/`  
- `scrap_hcky_ref_evenstrength.py` — scrape EV pages per season → `data/even_strength/`  
- **(Alternative to scraping)** `download_ref_hockey.py` — pull pre-scraped Standard + EV CSVs from S3 into `data/seasons/` and `data/even_strength/`  
- `build_ref_hockey.py` — normalize, fix Standard nTM/TOT, compute EV totals, merge → `data/outputs/`  
- `drop_goalies_etal_inplace.py` — drop goalies/zero-TOI, add EV minutes, drop `cf_rel`/`pos`, save final  
- *(Optional)* cap-hit merge → `data/outputs/hockeyref_final_with_cap.csv`  
- *(Optional)* DB load via `build_ref_hockey_data_table.py`


**Outputs**
- `data/outputs/hockeyref_std_concat.csv`  
- `data/outputs/hockeyref_even_concat.csv`  
- `data/outputs/hockeyref_final.csv` (after curation)


In [ ]:
from pathlib import Path

DATA = Path("data")
OUT = DATA / "outputs"
DATA.mkdir(exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print("Working dir:", Path.cwd())
print("Data dir:", DATA.resolve())
print("Outputs dir:", OUT.resolve())

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


In [ ]:
# Optional: instead of scraping locally, download pre-scraped Standard + EV CSVs from S3
# Auth: set AWS_PROFILE=nhl-beyond (or your profile) or have credentials configured.
# This writes into data/seasons and data/even_strength as used by build_ref_hockey.py
!python download_ref_hockey.py


In [ ]:
# Optional: regenerate Standard pages per season → data/seasons/
# !python scrap_hockey_ref_player.py --start 2013 --end 2024 --out data/seasons --log-level INFO


In [ ]:
# Optional: regenerate EV pages per season → data/even_strength/
# !python scrap_hcky_ref_evenstrength.py --start 2013 --end 2024 --out data/even_strength --log-level INFO


In [ ]:
# Produces:
# - data/outputs/hockeyref_std_concat.csv
# - data/outputs/hockeyref_even_concat.csv
# - data/outputs/hockeyref_final.csv
!python build_ref_hockey.py \
  --std-dir data/seasons \
  --even-dir data/even_strength \
  --out-dir data/outputs \
  --log-level INFO


In [ ]:
# Drops: goalies (by list), zero/NaN EV TOI, 'cf_rel', and 'pos'
# Adds: 'toi_even_strength_min' (season-total EV minutes, 2-dec string)
!python drop_goalies_etal_inplace.py


In [ ]:
import pandas as pd
cur = pd.read_csv("data/outputs/hockeyref_final.csv")
cur.head(10)


In [ ]:
## Optional: Cap-hit merge

- Normalize `(player, season)` join keys.  
- Clean cap hit to numeric USD (strip `$`, commas).  
- De-dup by `(player, season)` with median (robust to duplicates).


In [ ]:
from pathlib import Path
import pandas as pd

final_path = Path("data/outputs/hockeyref_final.csv")
cap_path   = Path("data/cap_hits/player_cap_hits.csv")  # adjust as needed

df = pd.read_csv(final_path)
cap = pd.read_csv(cap_path)

def norm_name(s): 
    return " ".join(str(s).lower().split())

for frame in (df, cap):
    if "player" in frame.columns:
        frame["player_norm"] = frame["player"].map(norm_name)
    if "season" in frame.columns:
        frame["season"] = frame["season"].astype(str).str.strip()

cap_ren = cap.rename(columns={"cap_hit":"cap_hit_usd"})
cap_ren["cap_hit_usd"] = (
    cap_ren["cap_hit_usd"].astype(str)
    .str.replace(r"[$,]", "", regex=True)
    .str.strip()
)
cap_ren["cap_hit_usd"] = pd.to_numeric(cap_ren["cap_hit_usd"], errors="coerce")

cap_agg = (
    cap_ren
    .dropna(subset=["cap_hit_usd"])
    .groupby(["player_norm","season"], as_index=False)["cap_hit_usd"].median()
)

merged = df.merge(cap_agg, on=["player_norm","season"], how="left").drop(columns=["player_norm"])
out_path = Path("data/outputs/hockeyref_final_with_cap.csv")
merged.to_csv(out_path, index=False)
print("Wrote", out_path, "rows:", len(merged))


## Optional: Load curated file into Postgres

Loads `data/outputs/hockeyref_final.csv` using a TEXT-first stage → destination table flow.

Requirements:
- `DATABASE_URL` (or the individual env vars used by `db_utils.py`)
- Quote mixed-case identifiers where applicable (e.g., `"CF%"`)


In [ ]:
# Creates/verifies stage & destination tables, COPYs, and replaces destination
!python build_ref_hockey_data_table.py


## Validation & diffs

- Use `diff_players_by_season.py` to find mismatches across sources/seasons.  
- Confirm EV totals look plausible after `mm:ss × GP` conversion (spot check extremes).


In [ ]:
# Example (adjust to your script’s CLI)
# !python diff_players_by_season.py \
#   --std data/outputs/hockeyref_std_concat.csv \
#   --even data/outputs/hockeyref_even_concat.csv \
#   --out data/outputs/_diagnostics


## Lessons learned

- Avg vs. total TOI: multiply only when parsing `mm:ss` per-game + GP.  
- Multi-team seasons: use `nTM` if present; otherwise synthesize consistent `TOT`.  
- Merge hygiene: normalize names/teams/seasons before grouping/joins.  
- Diagnostics early: write small samples to `_diagnostics/`.  
- Keep EV aggregation minimal (only for `(player, season)` merge).


## Repro checklist

- [ ] `data/`, `data/outputs/`, `data/goalies/` exist  
- [ ] `build_ref_hockey.py` produced 3 outputs in `data/outputs/`  
- [ ] `drop_goalies_etal_inplace.py` produced curated `hockeyref_final.csv`  
- [ ] (Optional) cap hits merged → `hockeyref_final_with_cap.csv`  
- [ ] (Optional) DB load complete (inspect with pgAdmin)  
- [ ] Logs reviewed (see `logs/`)


## Appendix: time parsing & helpers

- Regex patterns:
  - `_mmss_re` → per-game `mm:ss` (0–59 min)
  - `_mmmmss_re` → season totals `mmmm:ss`
  - `_hhmmss_re` → `HH:MM:SS` edge cases
- Functions (`time_utils.py`):
  - `compute_total_toi(df, toi_col="toi")` → adds `toi_seconds_total`, `toi_total_hms`
  - `seconds_to_hms(x)` → friendly `H:MM:SS`
